# Memory Selectivity Analysis — Personalization (PersonaMem-v2 / v31_best)

Mirrors `memory_analysis.ipynb` (cell 21) but for the preference-selectivity task.
Compares **untrained MemoryLLM-8B-chat** vs **v31_best** on a 200-example dataset of
2-preference conversation documents (1 held-out target + 1 non-held-out distractor).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))


In [ ]:
import os
# Pin this notebook to physical GPU 6 (or change to "7"). MUST be set before
# `import torch` — otherwise CUDA is already initialized on the visible set.
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

import random
import json
import re
from collections import defaultdict
import numpy as np
import torch
import torch.nn.functional as F
from gnm import MemoryLLMChatWrapper, build_model
from config_memoryllm_train import (
    MemoryllmTrainConfig, DistributedCfg, DataCfg, SamplingCfg,
    ModelCfg, LoopCtrlCfg, LoggingCfg,
)
import matplotlib.pyplot as plt
from scipy import stats

print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("torch.cuda.device_count() =", torch.cuda.device_count())
if torch.cuda.is_available():
    print("torch.cuda.get_device_name(0) =", torch.cuda.get_device_name(0))


# Helpers (verbatim copy from `memory_analysis.ipynb` cell 2)

In [ ]:
def find_answer_token_positions(wrapper, text: str, answer: str):
    """
    Find the token position(s) of the answer within a tokenized text.
    Returns list of (start_idx, end_idx) tuples for each occurrence.
    """
    # Tokenize the full text
    text_tokens = wrapper.tokenizer(text, return_tensors="pt", add_special_tokens=False)
    text_ids = text_tokens.input_ids[0].tolist()
    
    # Tokenize just the answer
    answer_tokens = wrapper.tokenizer(answer, return_tensors="pt", add_special_tokens=False)
    answer_ids = answer_tokens.input_ids[0].tolist()
    
    # Also try with leading space (common tokenization pattern)
    answer_with_space = " " + answer
    answer_space_tokens = wrapper.tokenizer(answer_with_space, return_tensors="pt", add_special_tokens=False)
    answer_space_ids = answer_space_tokens.input_ids[0].tolist()
    
    positions = []
    
    # Search for answer tokens in text
    for search_ids in [answer_ids, answer_space_ids]:
        for i in range(len(text_ids) - len(search_ids) + 1):
            if text_ids[i:i+len(search_ids)] == search_ids:
                positions.append((i, i + len(search_ids)))
    
    # Remove duplicates while preserving order
    seen = set()
    unique_positions = []
    for pos in positions:
        if pos not in seen:
            seen.add(pos)
            unique_positions.append(pos)
    
    return unique_positions, text_ids, answer_ids



@torch.no_grad()
def get_memory_delta_per_layer(
    wrapper: MemoryLLMChatWrapper,
    learning_instruction: str,
    document: str,
):
    """
    Returns memory delta for each layer.
    Output shape: [L, num_memory_tokens, hidden_dim]
    """
    wrapper.reset_memory()
    wrapper.model.eval()

    memory_context = (
        f"<|learning_instruction_start|>{learning_instruction}<|learning_instruction_end|>"
        f"<|document_start|>{document}<|document_end|>"
    )

    ctx_ids = wrapper.tokenizer(
        memory_context,
        return_tensors="pt",
        add_special_tokens=False
    ).input_ids.to(wrapper.model.device)

    # Get memory delta without committing
    delta = wrapper.model.inject_memory(ctx_ids, update_memory=False)
    
    # delta is [1, L, num_memory_tokens, hidden_dim]
    return delta[0].detach()  # [L, num_memory_tokens, hidden_dim]



@torch.no_grad()
def get_hidden_states_during_injection(wrapper, learning_instruction: str, document: str):
    """
    Get hidden states at ALL layers during the memory INJECTION forward pass.
    
    IMPORTANT: We call wrapper.model(...) with is_injection=True to get the 
    memory-influenced hidden states, NOT wrapper.model.model(...) which would
    bypass the memory machinery entirely.
    
    Returns: dict with 'hidden_states' tensor of shape [L+1, seq_len, D]
             where seq_len matches the input_ids length (memory tokens are internal)
    """
    wrapper.reset_memory()
    wrapper.model.eval()
    
    memory_context = (
        f"<|learning_instruction_start|>{learning_instruction}<|learning_instruction_end|>"
        f"<|document_start|>{document}<|document_end|>"
    )
    
    ctx_ids = wrapper.tokenizer(
        memory_context,
        return_tensors="pt",
        add_special_tokens=False
    ).input_ids.to(wrapper.model.device)
    
    # CRITICAL: Call the MemoryLLM forward with is_injection=True
    # This runs the actual memory injection forward pass where memory tokens
    # are prepended at each layer. The returned hidden_states correspond to
    # the input_ids positions (memory influence is internal).
    outputs = wrapper.model(
        input_ids=ctx_ids,
        output_hidden_states=True,
        is_injection=True,  # Memory write mode
        return_dict=True
    )
    
    # hidden_states is tuple of (L+1) tensors, each [1, seq_len, D]
    # These are the hidden states for the INPUT tokens (not memory tokens)
    # but they have been computed WITH memory tokens present in attention
    hidden_states = torch.stack([h.squeeze(0) for h in outputs.hidden_states], dim=0)
    
    return {
        'hidden_states': hidden_states.detach(),  # [L+1, seq_len, D]
        'input_ids': ctx_ids.squeeze(0).detach(),  # [seq_len]
        'memory_context': memory_context,  # for debugging
    }


# Model loading: untrained MemoryLLM-8B-chat + v31_best

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = MemoryllmTrainConfig(
    run_name    = "run",
    seed        = 42,
    distributed = DistributedCfg(cuda_visible_devices="0", nproc_per_node=4, master_port=29512),
    data        = DataCfg(),
    sampling    = SamplingCfg(),
    opt         = None,
    model       = ModelCfg(model='memoryllm_chat'),
    loop        = LoopCtrlCfg(),
    log         = LoggingCfg(),
)

print("Loading untrained MemoryLLM-8B-chat...")
untrained_wrapper = build_model(cfg)
untrained_wrapper.eval()
untrained_wrapper.to(device)
print("  done")

print("Loading v31_best...")
v31_wrapper = build_model(cfg)
V31_PATH = "../saved_models/preference_mc_v34/preference_mc_v34_best.pth"
sd = torch.load(V31_PATH, map_location='cpu', weights_only=False)
if isinstance(sd, dict) and "state_dict" in sd:
    sd = sd["state_dict"]
if isinstance(sd, dict) and "model" in sd:
    sd = sd["model"]
sd = {k.replace("module.", "", 1) if k.startswith("module.") else k: v for k, v in sd.items()}
missing, unexpected = v31_wrapper.load_state_dict(sd, strict=False)
print(f"  missing={len(missing)} unexpected={len(unexpected)}")
v31_wrapper.eval()
v31_wrapper.to(device)
print("  done")


# Dataset

Load the prebuilt 200-example dataset from
`data/analysis/personamem_memory_analysis.json` (built by
`build_personamem_memory_analysis_dataset.py`).
Each example: 1 held-out target preference + 1 non-held-out distractor preference,
regenerated as a 2-turn conversation document via `make_doc` from the original
training pipeline.

In [ ]:
DATASET_PATH = "../data/analysis/personamem_memory_analysis.json"
with open(DATASET_PATH) as f:
    examples = json.load(f)
print(f"loaded {len(examples)} examples")
print("keys:", list(examples[0].keys()))


# Sanity check: verify token spans on 2 examples

In [ ]:
for i in range(2):
    e = examples[i]
    print(f"\n--- Example {i} ---")
    print(f"target_topic={e['target_topic']!r:18s} distractor_topic={e['distractor_topic']!r}")
    print(f"target_answer:     {e['target_answer']}")
    print(f"distractor_answer: {e['distractor_answer']}")
    memory_context = (
        f"<|learning_instruction_start|>{e['learning_instruction']}<|learning_instruction_end|>"
        f"<|document_start|>{e['document']}<|document_end|>"
    )
    t_pos, ctx_ids, _ = find_answer_token_positions(untrained_wrapper, memory_context, e['target_answer'])
    d_pos, _, _ = find_answer_token_positions(untrained_wrapper, memory_context, e['distractor_answer'])
    print(f"target positions: {t_pos[:3]}")
    print(f"distractor positions: {d_pos[:3]}")
    if t_pos:
        s, ed = t_pos[0]
        print(f"target decoded:     {untrained_wrapper.tokenizer.decode(ctx_ids[s:ed])!r}")
    if d_pos:
        s, ed = d_pos[0]
        print(f"distractor decoded: {untrained_wrapper.tokenizer.decode(ctx_ids[s:ed])!r}")


# Memory Selectivity Index — function definitions (verbatim from `memory_analysis.ipynb` cell 21)

In [ ]:
METRIC_NAMES = ["dot", "cos_diff_mean", "two_cos_mean", "cos_diff_last", "two_cos_last"]
METRIC_DESC = {
    "dot":           r"$\langle d,\;\widehat{(\bar{t}-\bar{r})}\rangle$  (cell-21 original)",
    "cos_diff_mean": r"$\langle\hat{d},\;\widehat{(\bar{t}-\bar{r})}\rangle$  (cosine of mean-pooled difference)",
    "two_cos_mean":  r"$\langle\hat{d},\hat{\bar{t}}\rangle-\langle\hat{d},\hat{\bar{r}}\rangle$  (Option 1: two cosines, mean-pooled)",
    "cos_diff_last": r"$\langle\hat{d},\;\widehat{(t_{\text{last}}-r_{\text{last}})}\rangle$  (Option 2: last-token, cosine of difference)",
    "two_cos_last":  r"$\langle\hat{d},\hat{t}_{\text{last}}\rangle-\langle\hat{d},\hat{r}_{\text{last}}\rangle$  (Option 3: last-token, two cosines)",
}


def _unit(v):
    return F.normalize(v.unsqueeze(0), dim=1).squeeze(0)


@torch.no_grad()
def compute_memory_selectivity(wrapper, learning_instruction: str, document: str,
                                target_answer: str, distractor_answer: str):
    """One forward pass; five alignment metrics per layer. Returns a dict of
    np.array (one array of length L-1 per metric) or (None, error_str).
    """
    memory_context = (
        f"<|learning_instruction_start|>{learning_instruction}<|learning_instruction_end|>"
        f"<|document_start|>{document}<|document_end|>"
    )
    target_positions, ctx_ids, _ = find_answer_token_positions(wrapper, memory_context, target_answer)
    distractor_positions, _, _ = find_answer_token_positions(wrapper, memory_context, distractor_answer)
    if not target_positions or not distractor_positions:
        return None, "Answer not found"

    delta = get_memory_delta_per_layer(wrapper, learning_instruction, document)  # [L, 256, D]
    result = get_hidden_states_during_injection(wrapper, learning_instruction, document)
    hidden_states = result["hidden_states"]  # [L+1, seq_len, D]

    target_start, target_end = target_positions[0]
    distractor_start, distractor_end = distractor_positions[0]

    L = delta.shape[0]
    out = {k: [] for k in METRIC_NAMES}
    for layer in range(L - 1):
        h_idx = layer + 1
        t_span = hidden_states[h_idx, target_start:target_end, :].float()        # [Tt, D]
        r_span = hidden_states[h_idx, distractor_start:distractor_end, :].float() # [Tr, D]
        t_mean = t_span.mean(dim=0)
        r_mean = r_span.mean(dim=0)
        t_last = t_span[-1]
        r_last = r_span[-1]

        d = delta[layer].float().mean(dim=0).to(t_mean.device)
        d_hat = _unit(d)

        # M1, M2: cell-21 direction (mean-pooled difference).
        v_mean_hat = _unit(t_mean - r_mean)
        out["dot"].append((d @ v_mean_hat).item())
        out["cos_diff_mean"].append((d_hat @ v_mean_hat).item())

        # M3: two cosines, mean-pooled references.
        out["two_cos_mean"].append(
            (d_hat @ _unit(t_mean)).item() - (d_hat @ _unit(r_mean)).item()
        )

        # M4: cosine of difference, last-token references.
        v_last_hat = _unit(t_last - r_last)
        out["cos_diff_last"].append((d_hat @ v_last_hat).item())

        # M5: two cosines, last-token references.
        out["two_cos_last"].append(
            (d_hat @ _unit(t_last)).item() - (d_hat @ _unit(r_last)).item()
        )

    return {k: np.array(v) for k, v in out.items()}, None


# Collect selectivity scores for both models

Adapted from `collect_selectivity_scores` in cell 21: our dataset is a flat list
of dicts (each with `target_answer`/`distractor_answer`) rather than the nested
counterfact structure.

In [ ]:
def collect_flat(wrapper, examples, log_every=20):
    per_metric = {k: [] for k in METRIC_NAMES}
    errors = []
    for i, e in enumerate(examples):
        if i % log_every == 0:
            print(f"  {i}/{len(examples)}")
        out, err = compute_memory_selectivity(
            wrapper,
            learning_instruction=e['learning_instruction'],
            document=e['document'],
            target_answer=e['target_answer'],
            distractor_answer=e['distractor_answer'],
        )
        if err is not None:
            errors.append((i, err))
        else:
            for k in METRIC_NAMES:
                per_metric[k].append(out[k])
        if i % 10 == 9:
            torch.cuda.empty_cache()
    arrs = {k: np.stack(v, axis=0) if v else None for k, v in per_metric.items()}
    return arrs, errors

print("\nCollecting for UNTRAINED MemoryLLM-8B-chat...")
untrained_metrics, untrained_errors = collect_flat(untrained_wrapper, examples)
print(f"  collected={untrained_metrics['dot'].shape[0] if untrained_metrics['dot'] is not None else 0}  errors={len(untrained_errors)}")
torch.cuda.empty_cache()

# Free the untrained model from GPU before bringing v31 active to keep VRAM headroom.
untrained_wrapper.to('cpu')
torch.cuda.empty_cache()

print("\nCollecting for v31_best...")
v31_metrics, v31_errors = collect_flat(v31_wrapper, examples)
print(f"  collected={v31_metrics['dot'].shape[0] if v31_metrics['dot'] is not None else 0}  errors={len(v31_errors)}")
torch.cuda.empty_cache()


# Plot all five metrics side-by-side

Five panels in a 2×3 grid (last cell is the mid-layer summary bar chart). Each panel
is per-layer mean ± 1.96·SE for both models, computed on the same set of 200 examples
from a single forward pass — so any difference between panels is purely the choice of
alignment metric, not data variance.

In [ ]:
def per_layer_summary(arr):
    return arr.mean(axis=0), arr.std(axis=0) / np.sqrt(arr.shape[0])

summaries = {}
for k in METRIC_NAMES:
    u_mean, u_se = per_layer_summary(untrained_metrics[k])
    v_mean, v_se = per_layer_summary(v31_metrics[k])
    summaries[k] = {'u_mean': u_mean, 'u_se': u_se, 'v_mean': v_mean, 'v_se': v_se}

num_layers = len(summaries['dot']['u_mean'])
layers = list(range(num_layers))
mid_layers = slice(15, -1)

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()
for i, k in enumerate(METRIC_NAMES):
    ax = axes[i]
    s = summaries[k]
    ax.plot(layers, s['u_mean'], 'r-s', label='Untrained', linewidth=2, markersize=3)
    ax.fill_between(layers, s['u_mean']-1.96*s['u_se'], s['u_mean']+1.96*s['u_se'], alpha=0.2, color='red')
    ax.plot(layers, s['v_mean'], 'b-o', label='v31_best', linewidth=2, markersize=3)
    ax.fill_between(layers, s['v_mean']-1.96*s['v_se'], s['v_mean']+1.96*s['v_se'], alpha=0.2, color='blue')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Layer')
    ax.set_ylabel('Score')
    ax.set_title(f'M{i+1} — {k}\n' + METRIC_DESC[k], fontsize=10)
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)

# Final panel: mid-layer summary bar chart across all 5 metrics.
ax = axes[5]
x = np.arange(len(METRIC_NAMES))
width = 0.35
u_mids, u_mid_ses, v_mids, v_mid_ses = [], [], [], []
for k in METRIC_NAMES:
    u_arr = untrained_metrics[k][:, mid_layers].mean(axis=1)
    v_arr = v31_metrics[k][:, mid_layers].mean(axis=1)
    u_mids.append(u_arr.mean());      v_mids.append(v_arr.mean())
    u_mid_ses.append(u_arr.std()/np.sqrt(len(u_arr)))
    v_mid_ses.append(v_arr.std()/np.sqrt(len(v_arr)))
ax.bar(x - width/2, u_mids, width, yerr=[1.96*s for s in u_mid_ses], label='Untrained', color='red', alpha=0.7, capsize=4)
ax.bar(x + width/2, v_mids, width, yerr=[1.96*s for s in v_mid_ses], label='v31_best', color='blue', alpha=0.7, capsize=4)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(METRIC_NAMES, rotation=20, ha='right', fontsize=9)
ax.set_ylabel(f'Mean (layers {mid_layers.start}-{mid_layers.stop})')
ax.set_title(f'Mid-layer averages across metrics')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
PLOT_PATH = '../plots/memory_selectivity_personamem_5metrics.png'
plt.savefig(PLOT_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved 5-metric plot to {PLOT_PATH}')

# Stats per metric (paired t-test, Cohen's d) on mid-layer averages.
print(f"\nMid-layer slice {mid_layers}: per-metric stats (v31 vs untrained)")
print(f"{'metric':<18} {'u_mid':>8} {'v_mid':>8} {'t':>7} {'p':>9} {'d':>7}  {'pct_v>u':>8}")
stats_rows = {}
for k in METRIC_NAMES:
    u_arr = untrained_metrics[k][:, mid_layers].mean(axis=1)
    v_arr = v31_metrics[k][:, mid_layers].mean(axis=1)
    t_stat, p_val = stats.ttest_rel(v_arr, u_arr)
    diff = v_arr - u_arr
    cd = float(diff.mean()/diff.std()) if diff.std() > 0 else 0.0
    pct = float((v_arr > u_arr).mean()*100)
    stats_rows[k] = {'u_mid': float(u_arr.mean()), 'v_mid': float(v_arr.mean()),
                      't': float(t_stat), 'p': float(p_val), 'd': cd, 'pct_v_higher': pct}
    print(f"{k:<18} {u_arr.mean():>8.4f} {v_arr.mean():>8.4f} {t_stat:>7.2f} {p_val:>9.4g} {cd:>7.3f}  {pct:>7.1f}%")

JSON_PATH = '../evals/personamem/paper_results/memory_analysis/memory_selectivity_personalization.json'
with open(JSON_PATH, 'w') as f:
    json.dump({
        'metrics': {
            k: {
                'untrained_mean_per_layer': summaries[k]['u_mean'].tolist(),
                'untrained_se_per_layer':   summaries[k]['u_se'].tolist(),
                'v31_mean_per_layer':       summaries[k]['v_mean'].tolist(),
                'v31_se_per_layer':         summaries[k]['v_se'].tolist(),
                **stats_rows[k],
            }
            for k in METRIC_NAMES
        },
        'metric_descriptions': METRIC_DESC,
        'mid_layer_slice':      [mid_layers.start, mid_layers.stop],
        'num_examples':         int(untrained_metrics['dot'].shape[0]),
        'num_untrained_errors': len(untrained_errors),
        'num_v31_errors':       len(v31_errors),
        'dataset_path':         DATASET_PATH,
        'v31_checkpoint_path':  V31_PATH,
    }, f, indent=2)
print(f'Saved JSON to {JSON_PATH}')
